# Experiment: `dry_run`

**What varies:** Quick local sanity check — runs with DRY_RUN=True (5 epochs, 128 samples).

In [ ]:
# ============================================================
# EXPERIMENT CONFIG — the only section that changes between experiments
# ============================================================

EXPERIMENT_NAME = "dry_run"  # used for saving outputs
CONDITION_COL = "ramp_pattern_name"  # metadata column for grouping
USE_GLOBAL_ZSCORE = False
DRY_RUN = True

# --- Model architecture ---
model_kwargs = dict(
    hidden_dim=64,
    latent_dim=3,
)

# --- Training ---
training_kwargs = dict(
    lr=1e-3,
    epochs=300,
    batch_size=64,
    warmup=50,
    beta=1.0,      # KL weight (ramped from 0 over warmup epochs)
    patience=100,
    alpha=0.3,      # spectral loss weight (0.0 = MSE only)
)

# --- Evaluation ---
eval_kwargs = dict(
    n_recon_examples=12,
    n_traversal_steps=7,
    traversal_range=2.0,
    kl_active_threshold=0.05,
    dt=1.0,
)

---
# Setup & Data Loading

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, time
from datetime import datetime

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Device: {device} | DRY_RUN: {DRY_RUN}")

In [ ]:
from notebooks.experiment.preprocessing import load_and_clean, make_windows_sample, DEFAULT_STIM_COLS
from sklearn.model_selection import train_test_split

df = pd.read_parquet("dataset.parquet")
erk, stim, metadata = make_windows_sample(df, window_size=40, rng=42)

print(f"Windows: {erk.shape[0]}, Timepoints: {erk.shape[1]}")
print(f"Stim features: {stim.shape[1]} channels")
print(f"\nSamples per condition:")
print(metadata[CONDITION_COL].value_counts())

In [ ]:
# --- Train / val / test split (by cell, stratified by condition) ---
uid_meta = metadata.drop_duplicates("uid")[["uid", CONDITION_COL]]
uids_train, uids_test = train_test_split(
    uid_meta["uid"], test_size=0.2, random_state=42,
    stratify=uid_meta[CONDITION_COL],
)
uid_meta_train = uid_meta[uid_meta["uid"].isin(uids_train)]
uids_train, uids_val = train_test_split(
    uid_meta_train["uid"], test_size=0.125, random_state=42,
    stratify=uid_meta_train[CONDITION_COL],
)

mask_train = metadata["uid"].isin(uids_train).values
mask_val = metadata["uid"].isin(uids_val).values
mask_test = metadata["uid"].isin(uids_test).values

erk_train, stim_train = erk[mask_train], stim[mask_train]
erk_val, stim_val = erk[mask_val], stim[mask_val]
erk_test, stim_test = erk[mask_test], stim[mask_test]
meta_test = metadata[mask_test].reset_index(drop=True)

print(f"Train: {erk_train.shape[0]}  Val: {erk_val.shape[0]}  Test: {erk_test.shape[0]}")

In [ ]:
# --- Normalisation ---
erk_mu, erk_sigma = erk_train.mean(), erk_train.std()

if USE_GLOBAL_ZSCORE:
    erk_tr = (erk_train - erk_mu) / erk_sigma
    erk_va = (erk_val - erk_mu) / erk_sigma
    erk_te = (erk_test - erk_mu) / erk_sigma
    print(f"Using global z-score (mu={erk_mu:.4f}, sigma={erk_sigma:.4f})")
else:
    erk_tr, erk_va, erk_te = erk_train, erk_val, erk_test
    print("Using raw (baseline-normalised) ERK")

In [ ]:
# --- Dataset ---
class ERKDataset(Dataset):
    def __init__(self, erk, stim_features):
        erk_ch = torch.tensor(erk[:, np.newaxis, :])
        stim_ch = torch.tensor(stim_features)
        self.encoder_input = torch.cat([erk_ch, stim_ch], dim=1)
        self.stim_cond = stim_ch.clone()
        self.target = erk_ch.clone()

    def __len__(self):
        return self.encoder_input.shape[0]

    def __getitem__(self, idx):
        return self.encoder_input[idx], self.stim_cond[idx], self.target[idx]

train_ds = ERKDataset(erk_tr, stim_train)
val_ds = ERKDataset(erk_va, stim_val)
test_ds = ERKDataset(erk_te, stim_test)

if DRY_RUN:
    n_dry = 128
    train_ds = Subset(train_ds, range(min(n_dry, len(train_ds))))
    val_ds = Subset(val_ds, range(min(n_dry, len(val_ds))))
    test_ds = Subset(test_ds, range(min(n_dry, len(test_ds))))
    print(f"DRY RUN: {len(train_ds)} train, {len(val_ds)} val, {len(test_ds)} test")

BATCH_SIZE = training_kwargs["batch_size"]
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

enc_in, stim_cond, target = next(iter(train_loader))
print(f"Encoder input: {enc_in.shape}  Stim: {stim_cond.shape}  Target: {target.shape}")

---
# Model

In [ ]:
class Conv1dBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation=1):
        super().__init__()
        self.pad = ((kernel_size - 1) * dilation) // 2
        self.conv = nn.Conv1d(in_ch, out_ch, kernel_size, dilation=dilation)
        self.bn = nn.BatchNorm1d(out_ch)

    def forward(self, x):
        x = F.pad(x, (self.pad, self.pad))
        return F.gelu(self.bn(self.conv(x)))


class Encoder(nn.Module):
    def __init__(self, in_channels, hidden_dim=64, latent_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            Conv1dBlock(in_channels, hidden_dim, kernel_size=7),
            Conv1dBlock(hidden_dim, hidden_dim, kernel_size=5, dilation=2),
            Conv1dBlock(hidden_dim, hidden_dim, kernel_size=5, dilation=4),
            Conv1dBlock(hidden_dim, hidden_dim, kernel_size=3, dilation=8),
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

    def forward(self, x):
        h = self.net(x).mean(dim=-1)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder(nn.Module):
    def __init__(self, latent_dim=3, stim_channels=9, hidden_dim=64, seq_length=40):
        super().__init__()
        self.seq_length = seq_length
        self.z_proj = nn.Linear(latent_dim, hidden_dim)
        self.stim_net = nn.Sequential(
            Conv1dBlock(stim_channels, hidden_dim // 2, kernel_size=5),
            Conv1dBlock(hidden_dim // 2, hidden_dim, kernel_size=5),
        )
        self.decode_net = nn.Sequential(
            Conv1dBlock(hidden_dim * 2, hidden_dim, kernel_size=5),
            Conv1dBlock(hidden_dim, hidden_dim, kernel_size=5, dilation=2),
            Conv1dBlock(hidden_dim, hidden_dim // 2, kernel_size=5, dilation=4),
            nn.Conv1d(hidden_dim // 2, 1, kernel_size=1),
        )

    def forward(self, z, stim_cond):
        z_time = self.z_proj(z).unsqueeze(-1).expand(-1, -1, self.seq_length)
        s = self.stim_net(stim_cond)
        return self.decode_net(torch.cat([z_time, s], dim=1))


def spectral_loss(recon, target):
    return F.mse_loss(torch.abs(torch.fft.rfft(recon, dim=-1)),
                      torch.abs(torch.fft.rfft(target, dim=-1)))


class ConditionalBetaVAE(nn.Module):
    def __init__(self, in_channels, stim_channels, hidden_dim=64,
                 latent_dim=3, seq_length=40, beta=1.0, alpha=0.3):
        super().__init__()
        self.beta = beta
        self.alpha = alpha
        self.encoder = Encoder(in_channels, hidden_dim, latent_dim)
        self.decoder = Decoder(latent_dim, stim_channels, hidden_dim, seq_length)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def forward(self, encoder_input, stim_cond):
        mu, logvar = self.encoder(encoder_input)
        z = self.reparameterize(mu, logvar)
        recon = self.decoder(z, stim_cond)
        return recon, mu, logvar

    def loss(self, recon, target, mu, logvar):
        temporal = F.mse_loss(recon, target)
        spectral = spectral_loss(recon, target) if self.alpha != 0 else 0
        recon_loss = (1 - self.alpha) * temporal + self.alpha * spectral
        kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
        return recon_loss + self.beta * kl_loss, recon_loss, kl_loss

In [ ]:
# --- Instantiate model ---
n_stim = stim_train.shape[1]
seq_len = stim_train.shape[2]
in_ch = 1 + n_stim

model = ConditionalBetaVAE(
    in_channels=in_ch,
    stim_channels=n_stim,
    seq_length=seq_len,
    beta=training_kwargs["beta"],
    alpha=training_kwargs["alpha"],
    **model_kwargs,
).to(device)

with torch.no_grad():
    recon, mu, logvar = model(enc_in.to(device), stim_cond.to(device))
print(f"Recon: {recon.shape}, mu: {mu.shape}, logvar: {logvar.shape}")
print(f"Total params: {sum(p.numel() for p in model.parameters()):,}")

---
# Training

In [ ]:
def beta_schedule(epoch, warmup, target_beta):
    return min(target_beta, target_beta * epoch / max(1, warmup))

def train(model, train_loader, val_loader, cfg):
    optimizer = optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    best_val = float("inf")
    wait = 0
    history = {"train_total": [], "train_recon": [], "train_kl": [],
               "val_total": [], "val_recon": [], "val_kl": [], "beta": []}

    epochs = 5 if DRY_RUN else cfg["epochs"]
    warmup = 2 if DRY_RUN else cfg["warmup"]
    patience = 3 if DRY_RUN else cfg["patience"]
    target_beta = cfg["beta"]

    ckpt_path = f"best_model_{EXPERIMENT_NAME}.pt"

    for epoch in range(epochs):
        model.beta = beta_schedule(epoch, warmup, target_beta)
        history["beta"].append(model.beta)

        model.train()
        t_losses = []
        for enc_in, stim_cond, target in train_loader:
            enc_in, stim_cond, target = enc_in.to(device), stim_cond.to(device), target.to(device)
            recon, mu, logvar = model(enc_in, stim_cond)
            loss, recon_l, kl_l = model.loss(recon, target, mu, logvar)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            t_losses.append((loss.item(), recon_l.item(), kl_l.item()))

        model.eval()
        v_losses = []
        with torch.no_grad():
            for enc_in, stim_cond, target in val_loader:
                enc_in, stim_cond, target = enc_in.to(device), stim_cond.to(device), target.to(device)
                recon, mu, logvar = model(enc_in, stim_cond)
                loss, recon_l, kl_l = model.loss(recon, target, mu, logvar)
                v_losses.append((loss.item(), recon_l.item(), kl_l.item()))

        t = np.mean(t_losses, axis=0)
        v = np.mean(v_losses, axis=0)
        for i, k in enumerate(["total", "recon", "kl"]):
            history[f"train_{k}"].append(t[i])
            history[f"val_{k}"].append(v[i])

        scheduler.step(v[0])

        if v[0] < best_val:
            best_val = v[0]
            wait = 0
            torch.save(model.state_dict(), ckpt_path)
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}")
                break

        if epoch % 20 == 0:
            print(f"Epoch {epoch:3d} | β={model.beta:.3f} | "
                  f"Train: {t[0]:.4f} (r={t[1]:.4f} kl={t[2]:.4f}) | "
                  f"Val: {v[0]:.4f} (r={v[1]:.4f} kl={v[2]:.4f})")

    model.load_state_dict(torch.load(ckpt_path, weights_only=True))
    os.remove(ckpt_path)
    return history

In [ ]:
train_start = datetime.now()
t0 = time.time()

history = train(model, train_loader, val_loader, training_kwargs)

train_elapsed = time.time() - t0
print(f"\nTraining: {train_start.strftime('%Y-%m-%d %H:%M:%S')} | {train_elapsed:.0f}s ({train_elapsed/60:.1f} min)")

In [ ]:
# --- Training curves ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_total"], label="train")
axes[0].plot(history["val_total"], label="val")
axes[0].set_title("Total loss")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(history["train_recon"], label="train")
axes[1].plot(history["val_recon"], label="val")
axes[1].set_title("Reconstruction loss")
axes[1].set_xlabel("epoch")
axes[1].legend()

axes[2].plot(history["train_kl"][3:], label="train KL")
axes[2].plot(history["val_kl"][3:], label="val KL")
ax2 = axes[2].twinx()
ax2.plot(history["beta"], "k--", alpha=0.4, label="\u03b2")
ax2.set_ylabel("\u03b2")
axes[2].set_title("KL divergence + \u03b2 schedule")
axes[2].set_xlabel("epoch")
axes[2].legend(loc="upper left")
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

---
# Evaluation

In [ ]:
from eval_cvae import evaluate

result = evaluate(
    model=model,
    dataset=test_ds,
    metadata=meta_test,
    condition_col=CONDITION_COL,
    name=EXPERIMENT_NAME,
    **eval_kwargs,
)
result.summary()

In [ ]:
result.figures["reconstructions"]

In [ ]:
result.figures["mse_distribution"]

In [ ]:
result.figures["power_spectra"]

In [ ]:
result.figures["kl_per_dim"]

In [ ]:
result.figures["latent_traversals"]

In [ ]:
result.figures["stimulus_invariance"]

In [ ]:
result.figures["within_condition"]

In [ ]:
result.figures["encoder_uncertainty"]

---
# Save outputs

In [ ]:
OUTPUT_DIR = f"results/{EXPERIMENT_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Save model checkpoint with full metadata ---
start_str = train_start.strftime("%Y%m%d_%H%M%S")
model_save_path = os.path.join(OUTPUT_DIR, f"model_{start_str}.pt")

torch.save({
    "model_state_dict": model.state_dict(),
    "history": history,
    "config": {
        **model_kwargs,
        "alpha": training_kwargs["alpha"],
        "seq_length": seq_len,
        "n_stim_channels": n_stim,
        "in_channels": in_ch,
        "USE_GLOBAL_ZSCORE": USE_GLOBAL_ZSCORE,
    },
    "training_kwargs": training_kwargs,
    "train_start": train_start.isoformat(),
    "train_elapsed_s": train_elapsed,
    "erk_mu": float(erk_mu),
    "erk_sigma": float(erk_sigma),
    "device": str(device),
}, model_save_path)
print(f"Model saved to {model_save_path}")

# --- Save evaluation outputs ---
result.save(OUTPUT_DIR)
print(f"Evaluation saved to {OUTPUT_DIR}/")

---
# Notes

*Write observations here after running the experiment.*